In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ─── Cell 1: Install ───────────────────────────────────────────────────────────
!pip install imbalanced-learn --quiet

# ─── Cell 2: Imports & Config ──────────────────────────────────────────────────
import os, glob, gc
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from imblearn.over_sampling import SMOTE
from collections import defaultdict

input_folder  = '/content/drive/MyDrive/final_dataset/iot_cleaned_dedup'
output_folder = '/content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smote(hybrid30-30)'
bucket_dir    = '/content/buckets'
target_col    = 'label'
DOWN_TARGET   = 30_000
UP_TARGET     = 30_000

os.makedirs(output_folder, exist_ok=True)
os.makedirs(bucket_dir, exist_ok=True)
all_files = sorted(glob.glob(os.path.join(input_folder, "*.csv")))
print(f"Found {len(all_files)} files.")

# ─── Cell 3: Single-pass bucketing ─────────────────────────────────────────────
bucket_buffers = defaultdict(list)
FLUSH_EVERY = 200_000

label_to_file = {}

def get_bucket_path(label):
    if label not in label_to_file:
        idx = len(label_to_file)
        safe = f"class_{idx:03d}.parquet"
        label_to_file[label] = os.path.join(bucket_dir, safe)
    return label_to_file[label]

def flush_to_disk(label, chunks):
    path = get_bucket_path(label)
    new_df = pd.concat(chunks, ignore_index=True)
    if os.path.exists(path):
        existing = pd.read_parquet(path)
        new_df = pd.concat([existing, new_df], ignore_index=True)
    new_df.to_parquet(path, index=False)

print("Pass 1: Bucketing rows by class...")
for file in tqdm(all_files, desc="Reading CSVs"):
    df = pd.read_csv(file)
    for label, group in df.groupby(target_col):
        bucket_buffers[label].append(group)
        total_buffered = sum(len(g) for g in bucket_buffers[label])
        if total_buffered >= FLUSH_EVERY:
            flush_to_disk(label, bucket_buffers[label])
            bucket_buffers[label] = []
    del df
    gc.collect()

print("Flushing remaining buffers...")
for label, chunks in bucket_buffers.items():
    if chunks:
        flush_to_disk(label, chunks)
bucket_buffers.clear()

print(f"\nBucketed {len(label_to_file)} classes.")

# ─── Cell 4: Balance each class ────────────────────────────────────────────────
print("\nPass 2: Balancing each class...")

for label, bucket_path in tqdm(label_to_file.items(), desc="Balancing Classes"):
    class_df = pd.read_parquet(bucket_path)
    current_count = len(class_df)

    actual_label = class_df[target_col].iloc[0]
    print(f"\n  '{actual_label}': {current_count:,} rows")

    # ── BENIGN: keep all rows as-is, no sampling ──────────────────────────────
    if str(actual_label).upper() == 'BENIGN':
        final_df = class_df.copy()
        print(f"  → Kept as-is (BENIGN, {current_count:,} rows)")

    # CASE 1: Downsample
    elif current_count >= DOWN_TARGET:
        final_df = class_df.sample(n=DOWN_TARGET, random_state=42)
        print(f"  → Downsampled to {DOWN_TARGET:,}")

    # CASE 2: Too few for SMOTE
    elif current_count < 6:
        final_df = class_df.sample(n=UP_TARGET, replace=True, random_state=42)
        print(f"  → Random oversample to {UP_TARGET:,}")

    # CASE 3: SMOTE
    else:
        feature_cols = [c for c in class_df.columns if c != target_col]
        k_neighbors  = min(5, current_count - 1)

        X_real = class_df[feature_cols].select_dtypes(include=[np.number]).copy()
        X_real.replace([np.inf, -np.inf], np.nan, inplace=True)
        X_real.fillna(X_real.median(), inplace=True)
        X_real = X_real.clip(
            lower=np.finfo(np.float32).min / 2,
            upper=np.finfo(np.float32).max / 2
        ).astype('float32')

        dummy_df = X_real.sample(n=min(k_neighbors + 1, current_count), random_state=0).copy()

        X_combined = pd.concat([X_real, dummy_df], ignore_index=True)
        y_combined  = pd.Series(
            [actual_label] * len(X_real) + ['__DUMMY__'] * len(dummy_df)
        )

        try:
            sm = SMOTE(
                sampling_strategy={actual_label: UP_TARGET},
                k_neighbors=k_neighbors,
                random_state=42
            )
            X_res, y_res = sm.fit_resample(X_combined, y_combined)

            final_df = X_res[y_res == actual_label].copy()
            final_df[target_col] = actual_label
            print(f"  → SMOTE upsampled to {len(final_df):,}")

        except Exception as e:
            print(f"  ⚠ SMOTE failed ({e}), using random oversample.")
            final_df = class_df.sample(n=UP_TARGET, replace=True, random_state=42)

    # Save
    safe_label = str(actual_label).replace(' ', '_').replace('/', '-')
    out_path = os.path.join(output_folder, f"{safe_label}.csv")
    final_df.to_csv(out_path, index=False)
    print(f"  ✅ Saved → {out_path}")

    del class_df, final_df
    gc.collect()

print(f"\n✅ Done! Output: {output_folder}")

Found 62 files.
Pass 1: Bucketing rows by class...


Reading CSVs:   0%|          | 0/62 [00:00<?, ?it/s]

Flushing remaining buffers...

Bucketed 34 classes.

Pass 2: Balancing each class...


Balancing Classes:   0%|          | 0/34 [00:00<?, ?it/s]


  'DDOS-ICMP_FLOOD': 2,020,984 rows
  → Downsampled to 30,000
  ✅ Saved → /content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smote(hybrid30-30)/DDOS-ICMP_FLOOD.csv

  'DDOS-UDP_FLOOD': 1,967,818 rows
  → Downsampled to 30,000
  ✅ Saved → /content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smote(hybrid30-30)/DDOS-UDP_FLOOD.csv

  'DDOS-PSHACK_FLOOD': 1,730,270 rows
  → Downsampled to 30,000
  ✅ Saved → /content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smote(hybrid30-30)/DDOS-PSHACK_FLOOD.csv

  'DDOS-SYN_FLOOD': 1,827,885 rows
  → Downsampled to 30,000
  ✅ Saved → /content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smote(hybrid30-30)/DDOS-SYN_FLOOD.csv

  'DDOS-TCP_FLOOD': 1,611,482 rows
  → Downsampled to 30,000
  ✅ Saved → /content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smote(hybrid30-30)/DDOS-TCP_FLOOD.csv

  'DOS-UDP_FLOOD': 1,769,959 rows
  → Downsampled to 30,000
  ✅ Saved → /content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smote(hybrid30-30

In [ ]:
import pandas as pd
import glob
import os

# Set path to the new balanced folder
balanced_folder = '/content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smote(hybrid30-30)'
all_balanced_files = glob.glob(os.path.join(balanced_folder, "*.csv"))

distribution = {}

print("Calculating final distribution...")
for file in all_balanced_files:
    # We only need the length, but loading one row ensures the file is valid
    # and handles the case where the filename might not match the internal label perfectly
    df_temp = pd.read_csv(file)
    label = df_temp['label'].iloc[0] if not df_temp.empty else os.path.basename(file)
    distribution[label] = len(df_temp)

# Convert to Series for nice formatting
dist_series = pd.Series(distribution).sort_values(ascending=False)
print("\n" + "="*30)
print("FINAL CLASS DISTRIBUTION")
print("="*30)
print(dist_series)
print("="*30)
print(f"Total rows in balanced dataset: {dist_series.sum():,}")

Calculating final distribution...

FINAL CLASS DISTRIBUTION
BENIGN                     1026628
DDOS-ICMP_FLOOD              30000
DDOS-UDP_FLOOD               30000
DDOS-PSHACK_FLOOD            30000
DDOS-TCP_FLOOD               30000
DDOS-SYN_FLOOD               30000
DDOS-RSTFINFLOOD             30000
DDOS-SYNONYMOUSIP_FLOOD      30000
DOS-TCP_FLOOD                30000
DOS-UDP_FLOOD                30000
DOS-SYN_FLOOD                30000
MIRAI-GREETH_FLOOD           30000
MIRAI-UDPPLAIN               30000
MIRAI-GREIP_FLOOD            30000
DDOS-ICMP_FRAGMENTATION      30000
VULNERABILITYSCAN            30000
DDOS-ACK_FRAGMENTATION       30000
DDOS-UDP_FRAGMENTATION       30000
MITM-ARPSPOOFING             30000
BACKDOOR_MALWARE             30000
BROWSERHIJACKING             30000
COMMANDINJECTION             30000
DDOS-HTTP_FLOOD              30000
DDOS-SLOWLORIS               30000
DICTIONARYBRUTEFORCE         30000
DNS_SPOOFING                 30000
DOS-HTTP_FLOOD               3

In [ ]:
print("Checking for exact duplicates per class...")
print(f"{'Class Label':<30} | {'Total Rows':<12} | {'Unique Rows':<12} | {'Duplicates':<10}")
print("-" * 75)

for file in sorted(all_balanced_files):
    df_check = pd.read_csv(file)
    label = df_check['label'].iloc[0] if not df_check.empty else os.path.basename(file)

    total = len(df_check)
    # Check duplicates across all columns (including features)
    unique_count = len(df_check.drop_duplicates())
    duplicates = total - unique_count

    print(f"{str(label)[:30]:<30} | {total:<12,} | {unique_count:<12,} | {duplicates:<10,}")

    # Memory cleanup inside the loop
    del df_check
    gc.collect()

Checking for exact duplicates per class...
Class Label                    | Total Rows   | Unique Rows  | Duplicates
---------------------------------------------------------------------------
BACKDOOR_MALWARE               | 30,000       | 29,990       | 10        
BENIGN                         | 1,026,628    | 1,026,168    | 460       
BROWSERHIJACKING               | 30,000       | 29,897       | 103       
COMMANDINJECTION               | 30,000       | 29,867       | 133       
DDOS-ACK_FRAGMENTATION         | 30,000       | 29,999       | 1         
DDOS-HTTP_FLOOD                | 30,000       | 30,000       | 0         
DDOS-ICMP_FLOOD                | 30,000       | 29,951       | 49        
DDOS-ICMP_FRAGMENTATION        | 30,000       | 29,997       | 3         
DDOS-PSHACK_FLOOD              | 30,000       | 29,954       | 46        
DDOS-RSTFINFLOOD               | 30,000       | 29,899       | 101       
DDOS-SLOWLORIS                 | 30,000       | 30,000       | 0   

In [ ]:
import pandas as pd
import glob
import os
import gc
import numpy as np

balanced_folder = '/content/drive/MyDrive/final_dataset/iot_cleaned_sampled_smote(hybrid30-30)'
output_path = os.path.join(balanced_folder, 'merged.csv')

all_files = glob.glob(os.path.join(balanced_folder, "*.csv"))
# Exclude merged.csv if it already exists in the folder
all_files = [f for f in all_files if os.path.basename(f) != 'merged.csv']

print(f"Found {len(all_files)} class files to merge.")

dfs = []
for file in all_files:
    df = pd.read_csv(file)
    dfs.append(df)
    print(f"  Loaded: {os.path.basename(file)} → {len(df):,} rows")
    gc.collect()

print("\nConcatenating all classes...")
merged = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()

print(f"Total rows before shuffle: {len(merged):,}")

# --- Rigorous multi-pass shuffle ---
print("\nShuffling (pass 1/3)...")
merged = merged.sample(frac=1, random_state=42).reset_index(drop=True)

print("Shuffling (pass 2/3)...")
merged = merged.sample(frac=1, random_state=np.random.randint(0, 99999)).reset_index(drop=True)

print("Shuffling (pass 3/3)...")
merged = merged.sample(frac=1, random_state=np.random.randint(0, 99999)).reset_index(drop=True)

print(f"\nFinal label distribution:\n{merged['label'].value_counts()}")
print(f"\nTotal rows: {len(merged):,}")

print(f"\nSaving to {output_path} ...")
merged.to_csv(output_path, index=False)
print("✅ Done! Merged and shuffled CSV saved successfully.")

Found 34 class files to merge.
  Loaded: DDOS-ICMP_FLOOD.csv → 30,000 rows
  Loaded: DDOS-UDP_FLOOD.csv → 30,000 rows
  Loaded: DDOS-PSHACK_FLOOD.csv → 30,000 rows
  Loaded: DDOS-SYN_FLOOD.csv → 30,000 rows
  Loaded: DDOS-TCP_FLOOD.csv → 30,000 rows
  Loaded: DOS-UDP_FLOOD.csv → 30,000 rows
  Loaded: DDOS-RSTFINFLOOD.csv → 30,000 rows
  Loaded: DDOS-SYNONYMOUSIP_FLOOD.csv → 30,000 rows
  Loaded: DOS-TCP_FLOOD.csv → 30,000 rows
  Loaded: DOS-SYN_FLOOD.csv → 30,000 rows
  Loaded: BENIGN.csv → 1,026,628 rows
  Loaded: MIRAI-GREETH_FLOOD.csv → 30,000 rows
  Loaded: MIRAI-UDPPLAIN.csv → 30,000 rows
  Loaded: MIRAI-GREIP_FLOOD.csv → 30,000 rows
  Loaded: DDOS-ICMP_FRAGMENTATION.csv → 30,000 rows
  Loaded: VULNERABILITYSCAN.csv → 30,000 rows
  Loaded: DDOS-ACK_FRAGMENTATION.csv → 30,000 rows
  Loaded: DDOS-UDP_FRAGMENTATION.csv → 30,000 rows
  Loaded: MITM-ARPSPOOFING.csv → 30,000 rows
  Loaded: BACKDOOR_MALWARE.csv → 30,000 rows
  Loaded: BROWSERHIJACKING.csv → 30,000 rows
  Loaded: COMMANDI